## baseline:

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.model import CNNBaseline
from src.train import evaluate, fit, make_scheduler, train_one_epoch
from src.utils import get_device, set_seed


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
set_seed(SEED)
device = get_device()

IMG_SIZE = 128
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder('../data/processed/train', transform=base_transform)
val_ds = datasets.ImageFolder('../data/processed/val', transform=base_transform)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

In [ ]:
device= get_device()
x , y = next(iter(train_loader))
print(x.shape , y.shape)
model= CNNBaseline().to(device)
print(model(x.to(device)).shape)
print(sum(p.numel() for p in model.parameters()))

In [ ]:
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
m = CNNBaseline().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
for step in range(100):
    opt.zero_grad()
    loss = crit(m(x), y)
    loss.backward()
    opt.step()
    if step % 20 == 0:
        print(step, loss.item())

## training loop :

In [ ]:
EPOCHS = 20
model = CNNBaseline().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr=1e-3)
meta = {
    'architecture': 'CNNBaseline',
    'class_to_idx': train_ds.class_to_idx ,
    'IMG_SIZE' : IMG_SIZE,
    'seed': SEED,
    'config': { 'epochs': EPOCHS , 'lr': 1e-3 , 'batch_size': 32 , 'optimizer':'Adam'}}
history = fit(model , train_loader , val_loader , optimizer , criterion , device , EPOCHS , '../checkpoints/baseline.pt' , '../logs/baseline.csv' , meta = meta)